# CryptoPulse Sesión 1

Verificación funcional de PySpark con una muestra real de velas Spot de un minuto de BTCUSDT y SOLUSDT. Ejecuta todas las celdas. No se entrenan modelos ni se inicia Kafka en esta sesión.


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("cryptopulse-s01-verificacion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setJobGroup("cryptopulse-s01", "Lectura real BTC/SOL y verificación")
print("Spark:", spark.version)
print("Aplicación:", spark.sparkContext.applicationId)
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"))
spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/26 21:50:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 4.2.0
Aplicación: local-1793051449819
Zona horaria: UTC


## Cargar el CSV real

Los archivos de Binance desde 2025 usan microsegundos. El descargador ya convirtió los tiempos a ISO 8601 UTC sin perder precisión. El esquema conserva los precios y volúmenes como decimales.


In [2]:
from pathlib import Path
import json
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, LongType, BooleanType,
)

session_dir = Path("/opt/s01-arquitectura")
if not session_dir.exists():
    session_dir = Path.cwd()
data_dir = session_dir / "data"
manifest = json.loads((data_dir / "provenance.json").read_text(encoding="utf-8"))
assert manifest["synthetic"] is False
assert all(item["checksum_verified"] for item in manifest["archives"])

schema = StructType([
    StructField("symbol", StringType()),
    StructField("open_time", TimestampType()),
    StructField("open", DecimalType(28, 8)),
    StructField("high", DecimalType(28, 8)),
    StructField("low", DecimalType(28, 8)),
    StructField("close", DecimalType(28, 8)),
    StructField("volume", DecimalType(28, 8)),
    StructField("close_time", TimestampType()),
    StructField("quote_asset_volume", DecimalType(28, 8)),
    StructField("number_of_trades", LongType()),
    StructField("source", StringType()),
    StructField("is_closed", BooleanType()),
])

df = (
    spark.read
    .option("header", True)
    .option("mode", "FAILFAST")
    .option("timestampFormat", "yyyy-MM-dd\u0027T\u0027HH:mm:ss.SSSSSS\u0027Z\u0027")
    .schema(schema)
    .csv(str(data_dir / "cryptopulse_klines_muestra.csv"))
)
df


DataFrame[symbol: string, open_time: timestamp, open: decimal(28,8), high: decimal(28,8), low: decimal(28,8), close: decimal(28,8), volume: decimal(28,8), close_time: timestamp, quote_asset_volume: decimal(28,8), number_of_trades: bigint, source: string, is_closed: boolean]

In [3]:
df.printSchema()
df.orderBy("open_time", "symbol").show(5, truncate=False)


root
 |-- symbol: string (nullable = true)
 |-- open_time: timestamp (nullable = true)
 |-- open: decimal(28,8) (nullable = true)
 |-- high: decimal(28,8) (nullable = true)
 |-- low: decimal(28,8) (nullable = true)
 |-- close: decimal(28,8) (nullable = true)
 |-- volume: decimal(28,8) (nullable = true)
 |-- close_time: timestamp (nullable = true)
 |-- quote_asset_volume: decimal(28,8) (nullable = true)
 |-- number_of_trades: long (nullable = true)
 |-- source: string (nullable = true)
 |-- is_closed: boolean (nullable = true)

+-------+-------------------+--------------+--------------+--------------+--------------+-------------+--------------------------+------------------+----------------+------------------------+---------+
|symbol |open_time          |open          |high          |low           |close         |volume       |close_time                |quote_asset_volume|number_of_trades|source                  |is_closed|
+-------+-------------------+--------------+--------------+----

## Transformación mínima de la sesión 1

Seleccionar y mostrar columnas confirma que Spark ejecuta el trabajo. Las transformaciones analíticas se desarrollarán en S2.


In [4]:
df.select("symbol", "open_time", "close", "volume", "number_of_trades").orderBy(
    "open_time", "symbol"
).show(10, truncate=False)


+-------+-------------------+--------------+-------------+----------------+
|symbol |open_time          |close         |volume       |number_of_trades|
+-------+-------------------+--------------+-------------+----------------+
|BTCUSDT|2025-01-01 00:00:00|93610.93000000|8.21827000   |2631            |
|SOLUSDT|2025-01-01 00:00:00|189.71000000  |4904.06300000|1821            |
|BTCUSDT|2025-01-01 00:01:00|93652.00000000|12.14029000  |1273            |
|SOLUSDT|2025-01-01 00:01:00|189.71000000  |1321.02900000|744             |
|BTCUSDT|2025-01-01 00:02:00|93702.15000000|11.60597000  |1095            |
|SOLUSDT|2025-01-01 00:02:00|190.07000000  |7505.43900000|2406            |
|BTCUSDT|2025-01-01 00:03:00|93677.98000000|8.72958000   |1461            |
|SOLUSDT|2025-01-01 00:03:00|189.86000000  |1844.87300000|749             |
|BTCUSDT|2025-01-01 00:04:00|93661.20000000|5.24749000   |988             |
|SOLUSDT|2025-01-01 00:04:00|189.85000000  |642.51100000 |398             |
+-------+---

## Verificación funcional

Contrasta las filas leídas por Spark con la procedencia de los archivos y comprueba que Spark UI sirve la aplicación. El resultado se guarda solo después de superar las comprobaciones.


In [5]:
from datetime import datetime, timezone
import hashlib
import urllib.request
from pyspark.sql import functions as F

csv_hash = hashlib.sha256((data_dir / "cryptopulse_klines_muestra.csv").read_bytes()).hexdigest()
assert csv_hash == manifest["csv_sha256"], "El CSV cambió desde su descarga"
counts = {row["symbol"]: row["count"] for row in df.groupBy("symbol").count().collect()}
expected = {item["symbol"]: item["rows"] for item in manifest["archives"]}
assert counts == expected, (counts, expected)
assert df.filter(
    F.col("open_time").isNull() | F.col("close_time").isNull()
    | F.col("close").isNull() | F.col("is_closed").isNull()
    | ~F.col("is_closed")
).count() == 0, "Hay campos esenciales inválidos o velas sin cerrar"

with urllib.request.urlopen("http://127.0.0.1:4040/api/v1/applications", timeout=10) as response:
    applications = json.load(response)
assert any(item["id"] == spark.sparkContext.applicationId for item in applications)

verification = {
    "status": "PASS",
    "project": "CryptoPulse",
    "session": "S01",
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "application_id": spark.sparkContext.applicationId,
    "timezone": spark.conf.get("spark.sql.session.timeZone"),
    "rows_per_symbol": counts,
    "rows_total": sum(counts.values()),
    "csv_sha256": csv_hash,
    "spark_ui_verified": True,
    "spark_job_ids": spark.sparkContext.statusTracker().getJobIdsForGroup("cryptopulse-s01"),
}
artifacts_dir = session_dir / "artifacts"
artifacts_dir.mkdir(exist_ok=True)
(artifacts_dir / "verificacion_s01.json").write_text(
    json.dumps(verification, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(verification, ensure_ascii=False, indent=2))


{
  "status": "PASS",
  "project": "CryptoPulse",
  "session": "S01",
  "executed_at_utc": "2026-10-26T21:52:19.870528+00:00",
  "spark_version": "4.2.0",
  "spark_master": "local[*]",
  "application_id": "local-1793051449819",
  "timezone": "UTC",
  "rows_per_symbol": {
    "BTCUSDT": 1440,
    "SOLUSDT": 1440
  },
  "rows_total": 2880,
  "csv_sha256": "8e1612a5deb8374175069287c5fb4bed46486524dc09f4c168bb0bb7a6bc9aff",
  "spark_ui_verified": true,
  "spark_job_ids": [
    5,
    4,
    3,
    2,
    1,
    0
  ]
}


La sesión funcional termina cuando aparece `status: PASS`. En ejecución interactiva, mantén este kernel activo para consultar [Spark UI](http://localhost:4042). El laboratorio usa `local[*]`: ejecuta Spark en varios hilos de un solo contenedor y todavía no constituye un clúster de varios nodos.
